In [1]:
!pip uninstall -y langchain langchain-core langchain-openai langchain-community langchain-text-splitters networkx pypdf faiss-cpu fastapi uvicorn
!pip install langchain==0.1.17
!pip install langchain-openai==0.0.6
!pip install langchain-core==0.1.53 
!pip install langchain-community==0.0.33
!pip install networkx
!pip install pypdf
!pip install faiss-cpu
!pip install fastapi uvicorn

Found existing installation: langchain 0.1.17
Uninstalling langchain-0.1.17:
  Successfully uninstalled langchain-0.1.17
Found existing installation: langchain-core 0.1.53
Uninstalling langchain-core-0.1.53:
  Successfully uninstalled langchain-core-0.1.53
Found existing installation: langchain-openai 0.0.6
Uninstalling langchain-openai-0.0.6:
  Successfully uninstalled langchain-openai-0.0.6
Found existing installation: langchain-community 0.0.33
Uninstalling langchain-community-0.0.33:
  Successfully uninstalled langchain-community-0.0.33
Found existing installation: langchain-text-splitters 0.0.2
Uninstalling langchain-text-splitters-0.0.2:
  Successfully uninstalled langchain-text-splitters-0.0.2
Found existing installation: networkx 3.6.1
Uninstalling networkx-3.6.1:
  Successfully uninstalled networkx-3.6.1
Found existing installation: pypdf 6.9.2
Uninstalling pypdf-6.9.2:
  Successfully uninstalled pypdf-6.9.2
Found existing installation: faiss-cpu 1.13.2
Uninstalling faiss-cpu-

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.3 requires langchain-core<2.0.0,>=1.2.19, but you have langchain-core 0.1.53 which is incompatible.
langchain-classic 1.0.3 requires langchain-text-splitters<2.0.0,>=1.1.1, but you have langchain-text-splitters 0.0.2 which is incompatible.
langchain-mcp 0.2.1 requires langchain-core~=0.3.37, but you have langchain-core 0.1.53 which is incompatible.
langgraph-checkpoint 4.0.1 requires langchain-core>=0.2.38, but you have langchain-core 0.1.53 which is incompatible.
langgraph-prebuilt 1.0.8 requires langchain-core>=1.0.0, but you have langchain-core 0.1.53 which is incompatible.


  Using cached langchain_openai-0.0.6-py3-none-any.whl.metadata (2.5 kB)
Using cached langchain_openai-0.0.6-py3-none-any.whl (29 kB)
  Using cached langchain_community-0.0.33-py3-none-any.whl.metadata (8.5 kB)
Using cached langchain_community-0.0.33-py3-none-any.whl (1.9 MB)
  Attempting uninstall: langchain-community
    Found existing installation: langchain-community 0.0.38
    Uninstalling langchain-community-0.0.38:
      Successfully uninstalled langchain-community-0.0.38


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.1.17 requires langchain-community<0.1,>=0.0.36, but you have langchain-community 0.0.33 which is incompatible.


  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
Using cached networkx-3.6.1-py3-none-any.whl (2.1 MB)
  Using cached pypdf-6.9.2-py3-none-any.whl.metadata (7.1 kB)
Using cached pypdf-6.9.2-py3-none-any.whl (333 kB)
  Using cached faiss_cpu-1.13.2-cp312-cp312-win_amd64.whl.metadata (7.6 kB)
Using cached faiss_cpu-1.13.2-cp312-cp312-win_amd64.whl (18.9 MB)
  Using cached fastapi-0.135.2-py3-none-any.whl.metadata (28 kB)
  Using cached uvicorn-0.42.0-py3-none-any.whl.metadata (6.7 kB)
Using cached fastapi-0.135.2-py3-none-any.whl (117 kB)
Using cached uvicorn-0.42.0-py3-none-any.whl (68 kB)

   ---------------------------------------- 0/2 [uvicorn]
   -------------------- ------------------- 1/2 [fastapi]
   ---------------------------------------- 2/2 [fastapi]



ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastmcp 3.1.1 requires packaging>=24.0, but you have packaging 23.2 which is incompatible.


In [2]:
# If that still fails, ensure langchain is actually installed in your environment:
import langchain; print(langchain.__version__)
import langchain_core; print(langchain_core.__version__)

0.1.17
0.1.53


In [3]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.document_loaders import PyPDFLoader

In [4]:
# 2️⃣ Load PDF
loader = PyPDFLoader("HR_Policy_Manual_2023.pdf")
docs = loader.load()
print(f"Loaded {len(docs)} pages")

Loaded 208 pages


In [5]:
# 3️⃣ Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = splitter.split_documents(docs)
print(f"Created {len(chunks)} chunks")

Created 541 chunks


In [6]:
# 4️⃣ Create embeddings + FAISS index
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [7]:
# 5️⃣ Build RetrievalQA chain


# 5. Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 6. Build RetrievalQA chain
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",   # simple chain type
    retriever=vectorstore.as_retriever(),
    return_source_documents=True
)

In [8]:
# 6️⃣ Ask a question
query = "Summarize the key points from the HR policy manual"
result = qa.invoke(query)

print("Answer:", result["result"])
print("Sources:", result["source_documents"])

Answer: The key points from the IIMA HR Policy Manual 2023 are as follows:

1. The Manual compiles HR policies and procedures at IIMA and supersedes all previous documents on the same subjects.
2. The Institute reserves the right to interpret, change, suspend, or cancel any part of the Manual without notice, and will notify employees of such changes.
3. The preparation and maintenance of the Manual is the responsibility of the HR Department or a designated official.
4. The Manual is confidential and for restricted circulation only.
5. Policies in the Manual and any amendments will supersede existing policies.
6. Clarifications regarding the Manual can be sought from the HR Department.
7. For matters not covered in the Manual, the Institute will follow the rules and procedures prescribed by the Government of India.
8. Definitions include "Institute" referring to the Indian Institute of Management Ahmedabad and "Board" referring to the Board of Governors of the Institute.
Sources: [Docum

In [9]:
query = "What is casual leave"
result = qa.invoke(query)

print("Answer:", result["result"])
print("Sources:", result["source_documents"])

Answer: Casual leave is a type of leave that an employee is entitled to take for personal reasons or emergencies. According to the policy, an employee is allowed eight days of casual leave per calendar year, with the condition that not more than five days can be taken at a time. Casual leave can be combined with Special Casual leave but not with any other kind of leave. Sundays and holidays that fall during a period of casual leave are not counted as part of the leave. Additionally, casual leave can be taken for half a day, and it cannot be accumulated; any unused casual leave will lapse at the end of the calendar year.
Sources: [Document(page_content='(4) CANCELLATION OF LEAVE\n4.1 Cancellation of leave by the employee should be applied and approved by the supervisor. \n(5) KINDS OF LEAVE\n5.1 LEAVE TYPE 1: CASUAL LEAVE\n5.1.1 Casual leave admissible to an employee is eight days for a calendar year, subject to the \ncondition that not more than five days’ casual leave may be allowed a